In [8]:
from reinforced_lib import RLib 
import numpy as np 
from mapc_cmab.agents.additional_custom_agents.branching_dqn import BranchingDQN, BranchingQNetwork

aps = [0, 4, 8, 9, 10]

context = [0, 0, 0, 1, 0]

In [9]:
BEST_GROUPS = {
    0:  {0, 4, 9},
    4:  {0, 4, 10},
    8:  {4, 8, 10},
    9:  {0, 8, 9},
    10: {4, 9, 10},
}

In [10]:
def reward_fn(
    action: np.ndarray,
    sharing_position: int,
    optimal_group: set[int],
    aps: np.ndarray,
) -> float:

    # Safety: sharing AP must be selected.
    if action[sharing_position] != 1:
        return -100.0

    selected = set(
        aps[action.astype(bool)].tolist()
    )

    correct = len(selected & optimal_group)
    wrong = len(selected - optimal_group)
    missing = len(optimal_group - selected)

    return float(correct - wrong - missing)

In [11]:
def action_to_ap_group(action, aps):
    return np.asarray(aps)[action.astype(bool)]
    

In [14]:
import numpy as np
import optax

from reinforced_lib import RLib

APS = np.array(
    [0, 4, 8, 9, 10],
    dtype=np.int32,
)

NUM_APS = len(APS)

BEST_GROUPS = {
    0: {0, 4, 9},
    4: {0, 4, 10},
    8: {4, 8, 10},
    9: {0, 8, 9},
    10: {4, 9, 10},
}

AP_TO_POSITION = {
    int(ap): i
    for i, ap in enumerate(APS)
}


def make_context(sharing_position: int) -> np.ndarray:

    context = np.zeros(
        NUM_APS,
        dtype=np.float32,
    )

    context[sharing_position] = 1.0

    return context


def reward_fn(
    action: np.ndarray,
    sharing_position: int,
) -> float:

    # --------------------------------------------------------
    # Hard validity check
    # --------------------------------------------------------
    print(action)

    if action.shape != (NUM_APS,):
        raise ValueError(
            f"Bad action shape: {action.shape}"
        )


    # Sharing AP MUST be selected.
    if action[sharing_position] != 1:
        return -100.0

    sharing_ap = int(
        APS[sharing_position]
    )

    optimal_group = BEST_GROUPS[
        sharing_ap
    ]

    selected_group = set(
        APS[
            action.astype(bool)
        ].tolist()
    )

    correct = len(
        selected_group & optimal_group
    )

    wrong = len(
        selected_group - optimal_group
    )

    missing = len(
        optimal_group - selected_group
    )

    return float(
        correct
        - 20*wrong
        - 30*missing
    )


agent_params = {

    "q_network": BranchingQNetwork(
        num_branches=NUM_APS,
        hidden_dim=64,
    ),

    "obs_space_shape": (
        NUM_APS,
    ),

    "num_branches": NUM_APS,

    "optimizer": optax.adam(
        learning_rate=1e-3,
    ),

    "experience_replay_buffer_size": 5000,

    "experience_replay_batch_size": 64,

    "experience_replay_steps": 1,

    # Bandit-style test.
    "discount": 0.0,

    "epsilon": 1.0,

    "epsilon_decay": 0.995,

    "epsilon_min": 0.05,
}


rlib = RLib(
    agent_type=BranchingDQN,
    agent_params=agent_params,
    no_ext_mode=True,
)


agent_id = rlib.init(
    seed=42
)

rng = np.random.default_rng(
    42
)

previous_action = None
previous_reward = None

reward_history = []

for step in range(2000):

    sharing_position = int(
        rng.integers(NUM_APS)
    )

    observation = make_context(
        sharing_position
    )

    if step == 0:

        action = rlib.sample(
            agent_id=agent_id,
            is_training=True,

            sample_observations={
                "env_state": observation,
            },
        ).squeeze(0)

    else:

        action = rlib.sample(
            agent_id=agent_id,
            is_training=True,

            update_observations={
                "env_state": observation,
                "action": previous_action,
                "reward": previous_reward, 
                "terminal": False
            },
            
            sample_observations={
                "env_state": observation,
            },
        ).squeeze(0)


    # --------------------------------------------------------
    # Calculate reward
    # --------------------------------------------------------
    print(action)
    reward = reward_fn(
        action,
        sharing_position,
    )

    previous_action = action
    previous_reward = reward

    reward_history.append(
        reward
    )

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    if step % 100 == 0:

        epsilon = float(
            rlib
            ._agent_containers[
                agent_id
            ]
            .state
            .epsilon
        )

        sharing_ap = int(
            APS[sharing_position]
        )

        selected_group = np.asarray(
            action_to_ap_group(
                action,
                APS,
            )
        )

        optimal_group = sorted(
            BEST_GROUPS[
                sharing_ap
            ]
        )

        print(
            f"step={step:4d} | "
            f"sharing={sharing_ap:2d} | "
            f"epsilon={epsilon:.3f} | "
            f"action={action.tolist()} | "
            f"group={selected_group.tolist()} | "
            f"optimal={optimal_group} | "
            f"reward={reward:.1f}"
        )


# ============================================================
# GREEDY EVALUATION
# ============================================================

print()
print("=" * 70)
print("GREEDY EVALUATION")
print("=" * 70)


# For evaluation only, access the trained state and force epsilon=0.
agent_state = (
    rlib
    ._agent_containers[
        agent_id
    ]
    .state
)

agent_state = agent_state.replace(
    epsilon=0.0
)


for sharing_position in range(NUM_APS):

    observation = make_context(
        sharing_position
    )

    # Fixed evaluation key.
    import jax

    action = rlib._agent.sample(
        agent_state,
        jax.random.key(
            100 + sharing_position
        ),
        observation,
    ).squeeze(0)

    action = np.asarray(
        action,
        dtype=np.int32,
    )

    sharing_ap = int(
        APS[sharing_position]
    )

    predicted_group = np.asarray(
        action_to_ap_group(
            action,
            APS,
        )
    )

    optimal_group = np.array(
        sorted(
            BEST_GROUPS[
                sharing_ap
            ]
        ),
        dtype=np.int32,
    )

    print(
        f"sharing AP = {sharing_ap:2d} | "
        f"predicted = {predicted_group.tolist()} | "
        f"optimal = {optimal_group.tolist()} | "
        f"action = {action.tolist()}"
    )


# ============================================================
# FINAL STATISTICS
# ============================================================

print()
print(
    "Average reward, last 200 steps:",
    np.mean(
        reward_history[-200:]
    ),
)

print()
print(
    "Maximum possible reward:",
    max(
        len(group)
        for group in BEST_GROUPS.values()
    ),
)

[1 1 1 1 1]
[1 1 1 1 1]
step=   0 | sharing= 0 | epsilon=1.000 | action=[1, 1, 1, 1, 1] | group=[0, 4, 8, 9, 10] | optimal=[0, 4, 9] | reward=-37.0
[0 0 0 0 0]
[0 0 0 0 0]
[1 1 1 1 1]
[1 1 1 1 1]
[0 0 0 0 0]
[0 0 0 0 0]
[1 1 1 1 1]
[1 1 1 1 1]
[0 0 0 0 0]
[0 0 0 0 0]
[0 0 0 0 0]
[0 0 0 0 0]
[1 1 1 1 1]
[1 1 1 1 1]
[0 0 0 0 0]
[0 0 0 0 0]
[1 1 1 1 1]
[1 1 1 1 1]
[0 0 0 0 0]
[0 0 0 0 0]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[0 0 0 0 0]
[0 0 0 0 0]
[0 0 0 0 0]
[0 0 0 0 0]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[0 0 0 0 0]
[0 0 0 0 0]
[1 1 0 1 1]
[1 1 0 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[1 1 1 1 1]
[0 0 0 0 0]
[0 0 0 0 0]
[1 1 0 1 1]
[1 1 0 1 1]
[0 0 0 0 0]
[0 0 0 0 0]
[0 0 0 0 0]
[0 0 0 0 0]
[1 1 0 1 1]
[1 1 0 1 1]
[0 0 0 0 0]
[0 0 0 0 0]
[1 1 1 1 1]
[1 1 1 1 1]
[0 0 0 0 0]
[0 0 0 0 0]
[0 0 0 0 0]
